In [1]:
# --- MÓDULO DE INGESTÃO: EMPREGO SETORIAL ---

import pandas as pd

# Defino o ponto de entrada para os dados de vínculos formais exclusivos 
# do setor de mineração. Esta variável captura o "choque direto" de empregabilidade,
# permitindo isolar a mão de obra da atividade extrativa em relação ao município.
caminho_arquivo = r'C:\Users\fabio\TCC\8_Emprego_mineracao.csv'

try:
    # Realizo a ingestão do dataset do setor extrativo.
    # skiprows=2: Remoção do cabeçalho institucional.
    # encoding='latin-1': Decodificação mandatória para plataformas governamentais legadas.
    df_emprego_mineracao = pd.read_csv(
        caminho_arquivo,
        sep=';',
        skiprows=2,
        encoding='latin-1'
    )
    
    # DIAGNÓSTICO ESTRUTURAL INICIAL:
    # Confirmo se a carga pulou as linhas corretas e se as colunas estão alinhadas.
    print("--- Visualização do Dataset Setorial (Emprego Direto na Mineração) ---")
    print(df_emprego_mineracao.head())
    
    print("\n--- Inventário de Séries Temporais (Setor Extrativo) ---")
    print(df_emprego_mineracao.columns.tolist())

except Exception as e:
    print(f"ERRO DE INGESTÃO: Falha crítica ao carregar a matriz de Emprego Setorial: {e}")

--- TABELA CARREGADA ---
                  Município  2021  2020  2019  2018  2017  2016  2015  2014  \
0  RO-ALTA FLORESTA D OESTE    10     6     2     0     2    12    12     9   
1              RO-ARIQUEMES   365   325   325   387   416   418   392   683   
2                 RO-CABIXI     0     0     0     0     0     0     0     0   
3                 RO-CACOAL    62    70    66    75    97    81    79    65   
4             RO-CEREJEIRAS     0     0     0     0     0     0     0     0   

   2013  ...  2011  2010  2009  2008  2007  2006  2005  2004  2003  2002  
0    10  ...     7     9    10    12     9     9     8     0     6     2  
1   560  ...   518   336   288   211   200   248   224   149    60    65  
2     0  ...     0     0     0     0     0     0     0     0     0     0  
3   148  ...   140    58    56    42    24    24    22    48    13    15  
4     0  ...     0     2     3     2     3     4     3     1     3     1  

[5 rows x 21 columns]
--- COLUNAS ---
['Município

In [2]:
# --- ETAPA 2: PARSING DE STRINGS E EXTRAÇÃO DE CHAVES GEOGRÁFICAS ---

# Para garantir o alinhamento perfeito (Master Merge) com as bases do IBGE, 
# desmembro a string aglutinada fornecida pelo MTE ('UF-Município') em 
# duas dimensões isoladas. Isso é vital para a precisão do pareamento espacial.

print("Iniciando a separação e normalização das chaves geográficas...")

# 1. PARSING E EXPANSÃO DE VETOR (String Splitting)
# Utilizo a hifenização nativa da base da RAIS como delimitador para o split.
# n=1: Garante que apenas o primeiro hífen seja considerado, prevenindo erros 
# estruturais em municípios que possuem hífen na própria toponímia.
# expand=True: Converte o output de listas para colunas tabulares (Features).
colunas_separadas = df_emprego_mineracao['Município'].str.split('-', n=1, expand=True)

# 2. ATRIBUIÇÃO DE FEATURES DIMENSIONAIS
# Instancio as novas chaves geográficas seguindo a padronização de nomenclatura.
df_emprego_mineracao['Sigla_UF'] = colunas_separadas[0]
df_emprego_mineracao['Nome_Municipio'] = colunas_separadas[1]

# 3. LIMPEZA DE REDUNDÂNCIAS (Drop)
# Elimino a coluna original aglutinada para otimizar o uso de memória (RAM)
# e evitar a manutenção de informações geográficas duplicadas.
df_emprego_mineracao = df_emprego_mineracao.drop(columns=['Município'])

# 4. AUDITORIA DA NOVA ESTRUTURA
print("--- Diagnóstico: Extração de Features Geográficas ---")
print("Verificação da criação das variáveis 'Sigla_UF' e 'Nome_Municipio':")
print(df_emprego_mineracao.head())

--- Tabela com colunas separadas (UF e Município) ---
Veja as duas novas colunas no final:
   2021  2020  2019  2018  2017  2016  2015  2014  2013  2012  ...  2009  \
0    10     6     2     0     2    12    12     9    10     8  ...    10   
1   365   325   325   387   416   418   392   683   560   491  ...   288   
2     0     0     0     0     0     0     0     0     0     0  ...     0   
3    62    70    66    75    97    81    79    65   148   150  ...    56   
4     0     0     0     0     0     0     0     0     0     0  ...     3   

   2008  2007  2006  2005  2004  2003  2002  Sigla_UF         Nome_Municipio  
0    12     9     9     8     0     6     2        RO  ALTA FLORESTA D OESTE  
1   211   200   248   224   149    60    65        RO              ARIQUEMES  
2     0     0     0     0     0     0     0        RO                 CABIXI  
3    42    24    24    22    48    13    15        RO                 CACOAL  
4     2     3     4     3     1     3     1        RO    

In [3]:
# --- ETAPA 3: REESTRUTURAÇÃO PARA PAINEL DE DADOS SETORIAL (MELT) ---

# Para viabilizar a análise longitudinal do emprego direto na mineração,
# converto a matriz horizontal (Wide) em um painel vertical (Long).
# Este passo alinha a estrutura cronológica desta variável com as demais 
# bases do modelo (PIB, CFEM, Vínculos Totais).

print("Iniciando a transposição da série histórica de mineração...")

# 1. DEFINIÇÃO DAS CHAVES PRIMÁRIAS (ID_VARS)
# Utilizo as dimensões geográficas isoladas e normalizadas na etapa anterior.
chaves_geograficas = ['Sigla_UF', 'Nome_Municipio']

# 2. OPERAÇÃO DE TRANSPOSIÇÃO (WIDE-TO-LONG)
# Empilho as colunas temporais na variável 'Ano' e os quantitativos de 
# vínculos na variável métrica 'Empregos_mineracao'.
df_emprego_mineracao_final = df_emprego_mineracao.melt(
    id_vars=chaves_geograficas,
    var_name='Ano',
    value_name='Empregos_mineracao'
)

# 3. AUDITORIA DE ESTRUTURA E CRONOLOGIA
# Verifico a integridade do empilhamento analisando os extremos do novo painel.
print("--- Diagnóstico: Painel de Emprego Setorial (Long Format) ---")
print("Cabeçalho (Início da Série Histórica):")
print(df_emprego_mineracao_final.head())

print("\nRodapé (Período Mais Antigo/Fim do Empilhamento):")
print(df_emprego_mineracao_final.tail())

--- Tabela 'Derretida' (Formato Longo) ---
  Sigla_UF         Nome_Municipio   Ano  Empregos_mineracao
0       RO  ALTA FLORESTA D OESTE  2021                  10
1       RO              ARIQUEMES  2021                 365
2       RO                 CABIXI  2021                   0
3       RO                 CACOAL  2021                  62
4       RO             CEREJEIRAS  2021                   0

--- Últimas linhas (para ver os anos mais antigos) ---
       Sigla_UF Nome_Municipio   Ano  Empregos_mineracao
113095       GO        XAMBIOA  2002                   0
113096       GO       VILA BOA  2002                  14
113097       GO  VILA PROPICIO  2002                  18
113098       GO       IGNORADO  2002                   0
113099       DF       BRASILIA  2002                 322


In [4]:
# --- MÓDULO 3.2: INTEGRAÇÃO RETROATIVA DE SÉRIE HISTÓRICA (MINERAÇÃO 1996-2001) ---

import os
import numpy as np
import pandas as pd

# 1. CONFIGURAÇÃO DE DIRETÓRIO LEGADO
# Mapeamento dos microdados antigos para expansão da janela de observação 
# pré-tratamento do impacto minerário (Backcasting).
pasta_antigos = r'C:\Users\fabio\TCC\8_Emprego_mineracao_1996_2001'
lista_dfs_antigos = []

print(f"--- Iniciando Ingestão de Dados Legados (Setor Extrativo) ---")

# 2. PIPELINE DE EXTRAÇÃO E LIMPEZA DE ARQUIVOS HISTÓRICOS
for ano in range(1996, 2002):
    nome_arquivo = f'{ano}.csv'
    caminho_completo = os.path.join(pasta_antigos, nome_arquivo)
    
    try:
        # A. Ingestão com salto de cabeçalhos institucionais
        df_temp = pd.read_csv(caminho_completo, sep=';', encoding='latin-1', skiprows=1)
        
        # Referenciamento dinâmico de atributos
        col_cidade_suja = df_temp.columns[0] 
        col_valor = df_temp.columns[1]
        
        # B. Parsing e Extração de Chaves Geográficas (String Splitting)
        dados_separados = df_temp[col_cidade_suja].str.split('-', n=1, expand=True)
        
        df_temp['Sigla_UF'] = dados_separados[0]       
        df_temp['Nome_Municipio'] = dados_separados[1] 
        
        # C. Padronização Semântica e Injeção Temporal
        df_temp = df_temp.rename(columns={col_valor: 'Empregos_mineracao'})
        df_temp['Ano'] = ano 
        
        # D. Seleção de Schema (Feature Selection)
        df_limpo_ano = df_temp[['Sigla_UF', 'Nome_Municipio', 'Ano', 'Empregos_mineracao']]
        
        # TRATAMENTO DE EXCEÇÕES E IMPUTAÇÃO DE DADOS (Imputation)
        # Converte caracteres especiais (ex: hifens) gerados pela RAIS em NaN, imputando 0.
        # Racional Econômico: A omissão de dados setoriais reflete a inexistência de vínculos.
        df_limpo_ano['Empregos_mineracao'] = pd.to_numeric(df_limpo_ano['Empregos_mineracao'], errors='coerce').fillna(0)
        
        lista_dfs_antigos.append(df_limpo_ano)
        print(f"   -> Sucesso: Matriz do ano {ano} normalizada.")
        
    except Exception as e:
        print(f"   -> [ERRO CRÍTICO] Falha na leitura do lote {ano}: {e}")

# 3. CONSOLIDAÇÃO MESTRA DO PAINEL SETORIAL (1996-2021)

if lista_dfs_antigos:
    # Agregação do período legado
    df_antigos_consolidado = pd.concat(lista_dfs_antigos, ignore_index=True)
    
    # União Vertical (Stacking) com a série recente utilizando o namespace protegido
    df_emprego_mineracao_final = pd.concat([df_emprego_mineracao_final, df_antigos_consolidado], ignore_index=True)
    
    # Tipagem Forte (Data Type Casting)
    # Garante que o eixo temporal seja estritamente inteiro para viabilizar 
    # ordenação (Panel Sorting) e mesclagem (Master Merge) futuras.
    df_emprego_mineracao_final['Ano'] = df_emprego_mineracao_final['Ano'].astype(int)
    
    print("\n--- INTEGRAÇÃO DE SÉRIES (1996-2021) CONCLUÍDA ---")
    print

--- Processando arquivos antigos de mineração ---
-> Ano 1996 processado.
-> Ano 1997 processado.
-> Ano 1998 processado.
-> Ano 1999 processado.
-> Ano 2000 processado.
-> Ano 2001 processado.

--- UNIÃO 1996-2021 CONCLUÍDA ---
Total de linhas: 114966


C:\Users\fabio\AppData\Local\Temp\ipykernel_9444\1870254610.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo_ano['Empregos_mineracao'] = pd.to_numeric(df_limpo_ano['Empregos_mineracao'], errors='coerce').fillna(0)
C:\Users\fabio\AppData\Local\Temp\ipykernel_9444\1870254610.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_limpo_ano['Empregos_mineracao'] = pd.to_numeric(df_limpo_ano['Empregos_mineracao'], errors='coerce').fillna(0)
C:\Users\fabio\AppData\Local\Temp\ipykernel_9444\187025461

In [5]:
# --- ETAPA 3.5: ORDENAÇÃO E INDEXAÇÃO ESPAÇO-TEMPORAL (PANEL SORTING) ---

# A ordenação correta é um pré-requisito matemático para a estimação de 
# modelos em painel (Controle Sintético / Diferença em Diferenças). 
# Softwares econométricos exigem que a matriz esteja estruturada com 
# a unidade transversal (Município) agrupada e o eixo temporal (Ano) sequencial.

print("Iniciando o alinhamento longitudinal da série histórica...")

# 1. ORDENAÇÃO MULTIDIMENSIONAL
# by=['Nome_Municipio', 'Ano']: Estabelece uma hierarquia de ordenação geométrica.
# O algoritmo agrupa as séries transversalmente por município e, internamente, 
# ordena cronologicamente do período base ao período mais recente.
df_emprego_mineracao_ordenado = df_emprego_mineracao_final.sort_values(
    by=['Nome_Municipio', 'Ano']
)

# 2. AUDITORIA DA ORDENAÇÃO DO PAINEL
print("--- Diagnóstico: Matriz Setorial Ordenada (Painel Balanceado/Desbalanceado) ---")

# Correção Lógica de Pipeline: Como o Módulo 3.2 integrou a série retroativa (Backcasting),
# o período de largada da nossa matriz agora é 1996 (e não mais 2002).
print("Cabeçalho (Expectativa: Início da ordem alfabética municipal, ano-base 1996):")
print(df_emprego_mineracao_ordenado.head())

print("\nRodapé (Expectativa: Fim da ordem alfabética municipal, ano-limite 2021):")
print(df_emprego_mineracao_ordenado.tail())

--- Tabela Final ORDENADA ---
Primeiras linhas (deve mostrar o primeiro município e o primeiro ano, 2002):
       Sigla_UF   Nome_Municipio   Ano  Empregos_mineracao
112389      RS    PINTO BANDEIRA  2002                   0
106734      RS    PINTO BANDEIRA  2003                   0
101079      RS    PINTO BANDEIRA  2004                   0
95424       RS    PINTO BANDEIRA  2005                   0
89769       RS    PINTO BANDEIRA  2006                   0

Últimas linhas (ordenadas):
      Sigla_UF Nome_Municipio   Ano  Empregos_mineracao
27243       SC         ZORTEA  2017                   0
21588       SC         ZORTEA  2018                   0
15933       SC         ZORTEA  2019                   0
10278       SC         ZORTEA  2020                   0
4623        SC         ZORTEA  2021                   0


In [6]:
# --- ETAPA 4: SERIALIZAÇÃO E PERSISTÊNCIA DO PAINEL SETORIAL (OUTPUT) ---

# Concluo o processamento da variável de choque direto (Emprego na Mineração)
# exportando a matriz ordenada. Este arquivo isola o efeito da atividade extrativa
# e está rigorosamente estruturado para a integração final do modelo econométrico.

# 1. DEFINIÇÃO DO REPOSITÓRIO DE SAÍDA (REFINED ZONE)
# O roteamento para um diretório segregado ('FINALIZADOS') demonstra maturidade 
# na governança de dados do projeto, separando os artefatos brutos dos consolidados.
caminho_saida_csv = r'C:\Users\fabio\TCC\FINALIZADOS\08_Emprego_Min_FINAL.csv'

# 2. EXPORTAÇÃO E INTEGRIDADE ESTRUTURAL
# Invoco o dataframe protegido (df_emprego_mineracao_ordenado) e utilizo o 
# parâmetro index=False. Isso suprime a exportação dos índices do Pandas, 
# garantindo uma portabilidade perfeitamente limpa para o Stata ou R.
df_emprego_mineracao_ordenado.to_csv(
    caminho_saida_csv,
    index=False
)

print(f"--- PROTOCOLO DE EXPORTAÇÃO CONCLUÍDO ---")
print(f"Painel Setorial (Emprego Direto na Mineração 1996-2021) salvo com sucesso em:")
print(caminho_saida_csv)

--- SUCESSO! ---
Seu arquivo de emprego na mineração consolidado foi salvo em:
C:\Users\fabio\TCC\FINALIZADOS\08_Emprego_Min_FINAL.csv
